# Customer Churn Prediction & Retention Analytics

**Author:** Joston Mathabathe  
**Background:** BCom Business Management | Postgraduate Diploma in Data Science, Wits University  
**Tools:** Python, Pandas, NumPy, Matplotlib, Scikit-learn

## Business objective
Customer churn can reduce recurring revenue and increase the cost of acquiring replacement customers. The goal is to build an interpretable machine-learning workflow that helps a retention team identify customers who are more likely to churn and understand the business characteristics associated with churn.

This project combines **business thinking** with **data science**: translating a business problem into a prediction problem, analysing customer segments, preparing mixed data, comparing models, evaluating business-relevant metrics, interpreting drivers, and turning findings into retention actions.

> The model identifies customers with higher predicted churn risk. It does not prove that a customer will churn or that a retention action will cause a customer to stay.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


## 1. Dataset

This project uses IBM's public **Telco Customer Churn** sample dataset. It represents a fictional telecommunications company and contains customer demographics, tenure, services, contract and billing information, plus a `Churn` outcome.

The raw CSV is not stored in this repository. The notebook downloads it from IBM's archived public GitHub repository.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Data understanding

I inspect structure, data types, missing values, duplicates and target balance before modelling. This prevents an apparently accurate model from hiding poor performance on the smaller churn class.


In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("\nDuplicate customer IDs:", df["customerID"].duplicated().sum())
print("\nTarget distribution:")
display(df["Churn"].value_counts().to_frame("count"))
display((df["Churn"].value_counts(normalize=True)*100).round(2).to_frame("percent"))


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["ChurnFlag"] = df["Churn"].map({"No": 0, "Yes": 1})
df = df.drop_duplicates(subset="customerID").copy()
print("Cleaned shape:", df.shape)
print("TotalCharges missing:", df["TotalCharges"].isna().sum())


## 3. Exploratory business analysis

The first question is not "Which algorithm is best?" but **"Where is churn concentrated?"**

I examine churn across contract type, tenure, internet service, payment method and monthly charges. These views connect model development to customer-retention decisions.


In [ ]:
def churn_rate_by(column):
    out = (df.groupby(column)["ChurnFlag"].agg(["mean","count"])
             .rename(columns={"mean":"churn_rate","count":"customers"})
             .sort_values("churn_rate", ascending=False))
    out["churn_rate"] = (out["churn_rate"]*100).round(2)
    return out

display(churn_rate_by("Contract"))
display(churn_rate_by("InternetService"))
display(churn_rate_by("PaymentMethod"))


In [ ]:
df["TenureBand"] = pd.cut(df["tenure"], bins=[-1,6,12,24,48,np.inf],
                            labels=["0–6 months","7–12 months","13–24 months","25–48 months","49+ months"])
tenure_churn = churn_rate_by("TenureBand")
display(tenure_churn)
ax = tenure_churn["churn_rate"].plot(kind="bar", figsize=(8,4))
ax.set_title("Churn Rate by Tenure Band")
ax.set_xlabel("Tenure")
ax.set_ylabel("Churn rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
df.boxplot(column="MonthlyCharges", by="Churn", ax=axes[0])
axes[0].set_title("Monthly Charges by Churn")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Monthly charges")
df.boxplot(column="tenure", by="Churn", ax=axes[1])
axes[1].set_title("Tenure by Churn")
axes[1].set_xlabel("Churn")
axes[1].set_ylabel("Months")
plt.suptitle("")
plt.tight_layout()
plt.show()


## 4. Feature engineering

I create business-friendly features:
- `ServiceCount`: number of selected add-on services.
- `IsLongTermContract`: one- or two-year contract indicator.
- `HasInternet`: internet-service indicator.
- `HasPhone`: phone-service indicator.
- `TenureBand`: customer tenure segmentation.


In [ ]:
service_cols = ["OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
df["ServiceCount"] = df[service_cols].apply(lambda col: col.isin(["Yes"]).astype(int)).sum(axis=1)
df["IsLongTermContract"] = df["Contract"].isin(["One year","Two year"]).astype(int)
df["HasInternet"] = (df["InternetService"] != "No").astype(int)
df["HasPhone"] = (df["PhoneService"] == "Yes").astype(int)
display(df[["tenure","MonthlyCharges","TotalCharges","ServiceCount","IsLongTermContract","HasInternet","HasPhone","ChurnFlag"]].head())


## 5. Modelling strategy

This is a binary classification problem with `ChurnFlag` as the target.

I compare:
1. **Logistic Regression** — interpretable baseline.
2. **Random Forest** — nonlinear ensemble model.

The preprocessing pipeline handles numerical imputation/scaling and categorical one-hot encoding without leaking test-set information.


In [ ]:
target = "ChurnFlag"
feature_cols = [
    "gender","SeniorCitizen","Partner","Dependents","tenure","PhoneService",
    "MultipleLines","InternetService","OnlineSecurity","OnlineBackup",
    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies",
    "Contract","PaperlessBilling","PaymentMethod","MonthlyCharges","TotalCharges",
    "ServiceCount","IsLongTermContract","HasInternet","HasPhone"
]
X = df[feature_cols].copy()
y = df[target].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
numeric_features = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["int64","float64"]).columns.tolist()

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num",numeric_pipe,numeric_features),("cat",categorical_pipe,categorical_features)])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
])
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=5,
                                     class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))
])
logistic_model.fit(X_train,y_train)
rf_model.fit(X_train,y_train)


In [ ]:
def evaluate_model(name, model):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test,pred),
        "Precision": precision_score(y_test,pred,zero_division=0),
        "Recall": recall_score(y_test,pred,zero_division=0),
        "F1": f1_score(y_test,pred,zero_division=0),
        "ROC-AUC": roc_auc_score(y_test,prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", logistic_model),
    evaluate_model("Random Forest", rf_model)
]).set_index("Model")
display(results.round(3))


## 6. Business-focused evaluation

For a retention team, a false negative means a customer who actually churns was not flagged. Missing these customers can reduce retention opportunities.

I therefore evaluate:
- **Recall** — how many churners are identified.
- **Precision** — how many flagged customers actually churn.
- **F1** — balance between precision and recall.
- **ROC-AUC** — ranking/discrimination ability.
- **Accuracy** — overall correctness, but not the only metric.


In [ ]:
best_model_name = results["ROC-AUC"].idxmax()
best_model = logistic_model if best_model_name == "Logistic Regression" else rf_model
pred = best_model.predict(X_test)
prob = best_model.predict_proba(X_test)[:,1]
print("Selected model for interpretation:", best_model_name)
print(classification_report(y_test,pred,target_names=["No Churn","Churn"],zero_division=0))


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=["No Churn","Churn"],ax=ax)
ax.set_title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6,5))
RocCurveDisplay.from_predictions(y_test,prob,ax=ax)
ax.set_title(f"ROC Curve — {best_model_name}")
plt.tight_layout()
plt.show()


## 7. Model interpretation

A retention model is more useful when analysts can explain which customer characteristics are associated with higher predicted risk.

For Logistic Regression, coefficients provide directional interpretation after preprocessing. For Random Forest, feature importance indicates which transformed features contribute most to model decisions.

These are associations for prioritisation, not proof of causation.


In [ ]:
fitted_preprocessor = best_model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()

if best_model_name == "Logistic Regression":
    values = best_model.named_steps["model"].coef_[0]
    importance = pd.DataFrame({"feature":feature_names,"value":values,"abs_value":np.abs(values)}).sort_values("abs_value",ascending=False)
    display(importance.head(15)[["feature","value"]])
else:
    values = best_model.named_steps["model"].feature_importances_
    importance = pd.DataFrame({"feature":feature_names,"importance":values}).sort_values("importance",ascending=False)
    display(importance.head(15))


## 8. Retention prioritisation

Prediction becomes useful when it supports a business process.

Illustrative risk bands:
- **High risk:** predicted probability ≥ 0.70
- **Medium risk:** 0.40–0.69
- **Lower risk:** < 0.40

In a real business, thresholds should be selected using campaign capacity, contact cost, customer value and the relative cost of false negatives and false positives.


In [ ]:
scored = X_test.copy()
scored["ActualChurn"] = y_test.values
scored["ChurnProbability"] = prob
scored["RiskSegment"] = pd.cut(
    scored["ChurnProbability"],
    bins=[-0.001,0.40,0.70,1.001],
    labels=["Lower risk","Medium risk","High risk"]
)
risk_summary = (scored.groupby("RiskSegment",observed=False)
                .agg(Customers=("ActualChurn","size"),
                     ActualChurnRate=("ActualChurn","mean"),
                     AveragePredictedRisk=("ChurnProbability","mean")))
risk_summary["ActualChurnRate"] = (risk_summary["ActualChurnRate"]*100).round(2)
risk_summary["AveragePredictedRisk"] = (risk_summary["AveragePredictedRisk"]*100).round(2)
display(risk_summary)


## 9. Business recommendations

1. **Prioritise high-risk customers** for proactive retention review.
2. **Segment the intervention** using contract, tenure, services and payment characteristics rather than one generic offer.
3. **Combine churn probability with customer value** so limited retention resources are allocated thoughtfully.
4. **Track outcomes after intervention** using churn, response rate, offer cost and customer value retained.
5. **Run controlled tests** where possible to determine whether an intervention actually changes retention.

### Suggested retention dashboard KPIs
- High-risk customers
- High-risk percentage
- Customers contacted
- Offer/contact acceptance rate
- Churn rate among contacted customers
- Retention campaign cost
- Revenue/customer value retained
- Model recall and precision


## 10. Limitations and responsible use

- The dataset is a fictional IBM sample, not a live company dataset.
- It is a cross-sectional snapshot, so this should not be presented as proof of future-month forecasting.
- The churn label represents observed churn, not the effect of a retention intervention.
- Production systems should use time-based validation when historical snapshots are available.
- Model probabilities should be calibrated before being treated as business probabilities.
- Customer decisions should consider privacy, fairness, business rules and applicable regulation.
- Sensitive or protected attributes should not be used for unfair discrimination.
- The project demonstrates analytical capability; it is not a production-ready deployment.

## Final takeaway

The project demonstrates an end-to-end workflow from **business problem → data preparation → exploratory analysis → machine learning → model evaluation → customer-risk segmentation → business recommendations**.
